In [3]:
from datasets import load_dataset

dataset = load_dataset("corto-ai/handwritten-text")

print(dataset)


README.md:   0%|          | 0.00/528 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6482 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2915 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'image'],
        num_rows: 6482
    })
    valid: Dataset({
        features: ['text', 'image'],
        num_rows: 976
    })
    test: Dataset({
        features: ['text', 'image'],
        num_rows: 2915
    })
})


In [4]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_name = "microsoft/trocr-small-handwritten"

processor = TrOCRProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/246M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import numpy as np
import torch

def preprocess(examples):
    # Convert PIL images to model pixel format
    pixel_values = processor(examples["image"], return_tensors="pt").pixel_values

    # Encode text labels
    with processor.as_target_processor():
        labels = processor(examples["text"], padding="max_length", max_length=128, truncation=True).input_ids

    # Replace padding token id by -100 to ignore during loss
    labels = [[(l if l != processor.tokenizer.pad_token_id else -100) for l in label] for label in labels]

    examples["pixel_values"] = pixel_values
    examples["labels"] = labels
    return examples

processed_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)


Map:   0%|          | 0/6482 [00:00<?, ? examples/s]

ValueError: Unsupported number of image dimensions: 2

In [6]:

import numpy as np
import torch

def preprocess(examples):
    # Convert PIL images to model pixel format
    # Ensure images have 3 channels (convert grayscale to RGB if necessary)
    processed_images = [image.convert("RGB") if image.mode != "RGB" else image for image in examples["image"]]
    pixel_values = processor(processed_images, return_tensors="pt").pixel_values

    # Encode text labels
    with processor.as_target_processor():
        labels = processor(examples["text"], padding="max_length", max_length=128, truncation=True).input_ids

    # Replace padding token id by -100 to ignore during loss
    labels = [[(l if l != processor.tokenizer.pad_token_id else -100) for l in label] for label in labels]

    examples["pixel_values"] = pixel_values
    examples["labels"] = labels
    return examples

processed_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/6482 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:118: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your images inputs, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/976 [00:00<?, ? examples/s]

Map:   0%|          | 0/2915 [00:00<?, ? examples/s]

In [7]:
def collate_fn(batch):
    pixel_values = torch.stack([torch.tensor(b["pixel_values"]) for b in batch])
    labels = torch.tensor([b["labels"] for b in batch])
    return {"pixel_values": pixel_values, "labels": labels}


In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-finetuned-iam",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    fp16=True,
    save_steps=2000,
    eval_steps=500,
    logging_steps=200,
    num_train_epochs=5,
)

In [11]:
!pip install jiwer
from jiwer import wer
import editdistance

def cer(ref, hyp):
    ref = ref.replace(" ", "")
    hyp = hyp.replace(" ", "")
    dist = editdistance.eval(ref, hyp)
    return dist / max(1, len(ref))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = processor.batch_decode(logits, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    refs = processor.batch_decode(labels, skip_special_tokens=True)

    cer_scores = [cer(r, p) for r, p in zip(refs, preds)]
    wer_scores = [wer(r, p) for r, p in zip(refs, preds)]

    return {"cer": sum(cer_scores) / len(cer_scores),
            "wer": sum(wer_scores) / len(wer_scores)}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.7 MB/s eta 0:00:00


In [13]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["valid"],
    data_collator=collate_fn,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:139: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(
/tmp/ipython-input-261850693.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


ValueError: API key must be at least 40 characters long, yours was 11